In [2]:
# ==============================================================================
# SiliGuide: Custom Siling Labuyo Leaf Instance Segmentation Pipeline
# Model: YOLO26n Instance Segmentation (Ultralytics)
#
# A clean, practical, and reproducible step-by-step training pipeline for
# segmenting Siling Labuyo (chili pepper) leaves using YOLO26n.
#
# Pipeline Workflow:
# 1. Environment & hardware setup
# 2. Configuration & path definitions
# 3. Automatic class discovery from Roboflow data.yaml
# 4. Dataset discovery & polygon annotation validation
# 5. Deterministic 70/15/15 dataset splitting & leakage check
# 6. Class instance counts & initial distribution
# 7. Targeted class-balancing data augmentation (Optional)
# 8. YOLO dataset folder structure & data.yaml creation
# 9. YOLO26n Model initialization & training configuration
# 10. Training execution
# 11. Validation & test evaluation (Accuracy, Precision, Recall, F1, mAP, Confusion Matrix)
# 12. Test prediction visualization
# ==============================================================================

import os
import sys
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import cv2

import torch
import ultralytics
from ultralytics import YOLO

# 1. Check Python and Library Versions
print("=" * 60)
print("ENVIRONMENT & HARDWARE SUMMARY")
print("=" * 60)
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"CUDA Ready  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device  : {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU mode.")
print("=" * 60)

# 2. Set Random Seed for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[REPRODUCIBILITY] Random seed locked to {SEED}.")

ENVIRONMENT & HARDWARE SUMMARY
Python      : 3.10.0
PyTorch     : 2.5.1+cu121
Ultralytics : 8.4.131
CUDA Ready  : True
GPU Device  : NVIDIA GeForce RTX 2050
[REPRODUCIBILITY] Random seed locked to 42.


In [3]:
# ==============================================================================
# PIPELINE CONFIGURATION
# Set all dataset directories, split ratios, augmentation toggles, and model parameters.
# ==============================================================================

# Source directory containing images, labels, and data.yaml
SOURCE_DIR = Path("siliguide_dataset.yolo26_v3") if Path("siliguide_dataset.yolo26_v3").exists() else Path(".")

# Output directory for the freshly structured YOLO dataset
OUTPUT_DIR = Path("labuyo_yolo_dataset_v5")

# Output directory for training checkpoints and experiment logs
RUNS_DIR = Path("labuyo_segmentation_v6")
EXPERIMENT_NAME = "yolo26n_labuyo_leaf_seg_v6"

# Split Ratios (70% Train, 15% Validation, 15% Test)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

assert np.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0), "Split ratios must sum to 1.0"

# Target YOLO26n Segmentation Model File
if Path("yolo26n-seg.pt").exists():
    MODEL_PATH = "yolo26n-seg.pt"
elif (SOURCE_DIR / "yolo26n-seg.pt").exists():
    MODEL_PATH = str(SOURCE_DIR / "yolo26n-seg.pt")
else:
    MODEL_PATH = "yolo26n-seg.pt"

# Targeted Augmentation Settings (Applies ONLY to training split to balance minority classes)
ENABLE_TARGETED_AUGMENTATION = False   # Set to True to balance minority classes, False to skip
TARGET_BALANCE_RATIO        = 1.0    # 1.0 means augment minority classes up to the majority class count

print("Pipeline Configuration:")
print(f" - Source Folder        : {SOURCE_DIR.resolve()}")
print(f" - Output Folder        : {OUTPUT_DIR.resolve()}")
print(f" - Splits               : Train={TRAIN_RATIO:.0%}, Val={VAL_RATIO:.0%}, Test={TEST_RATIO:.0%}")
print(f" - Targeted Augmentation: {'ENABLED' if ENABLE_TARGETED_AUGMENTATION else 'DISABLED'}")
print(f" - Model Architecture   : {MODEL_PATH}")

Pipeline Configuration:
 - Source Folder        : C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\siliguide_dataset.yolo26_v3
 - Output Folder        : C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5
 - Splits               : Train=70%, Val=15%, Test=15%
 - Targeted Augmentation: DISABLED
 - Model Architecture   : yolo26n-seg.pt


In [4]:
# ==============================================================================
# AUTOMATIC CLASS DISCOVERY FROM data.yaml
# Reads class definitions directly from the Roboflow dataset configuration.
# ==============================================================================

def load_class_names(source_path: Path) -> dict:
    """Finds and parses class names from existing data.yaml or falls back to defaults."""
    yaml_files = list(source_path.glob("*.yaml")) + list(source_path.glob("*.yml")) + list(Path(".").glob("*.yaml"))
    
    for yf in yaml_files:
        try:
            with open(yf, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
                if isinstance(data, dict) and "names" in data:
                    names = data["names"]
                    if isinstance(names, list):
                        return {i: str(name) for i, name in enumerate(names)}
                    elif isinstance(names, dict):
                        return {int(k): str(v) for k, v in names.items()}
        except Exception as e:
            print(f"[WARN] Error reading {yf}: {e}")
            
    # Default Siling Labuyo classes if no YAML is found
    return {0: "curling", 1: "healthy", 2: "spots", 3: "yellowing"}

CLASS_NAMES = load_class_names(SOURCE_DIR)
NUM_CLASSES = len(CLASS_NAMES)

print(f"[CLASSES] Detected {NUM_CLASSES} classes:")
for class_id, class_name in CLASS_NAMES.items():
    print(f"   ID {class_id} -> {class_name}")

[CLASSES] Detected 4 classes:
   ID 0 -> curling
   ID 1 -> healthy
   ID 2 -> spots
   ID 3 -> yellowing


In [5]:
# ==============================================================================
# DATASET DISCOVERY & ANNOTATION VALIDATION
# Scans for all image files and validates their corresponding YOLO polygon labels:
# - Validates image file readability.
# - Validates polygon coordinates are normalized between 0.0 and 1.0.
# - Validates minimum vertices (at least 3 coordinate pairs = 6 numbers).
# - Validates class IDs match the discovered classes.
# ==============================================================================

VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}

def discover_and_validate(source_path: Path, class_dict: dict):
    # Traverse safely, skipping environment and run directories
    ignore_dirs = {".venv", ".venv-linux", ".git", "runs", "__pycache__", ".ipynb_checkpoints", "labuyo_yolo_dataset"}
    all_files = []
    for root, dirs, files in os.walk(source_path):
        dirs[:] = [d for d in dirs if d not in ignore_dirs and not d.startswith(".")]
        for f in files:
            all_files.append(Path(root) / f)

    image_paths = [p for p in all_files if p.is_file() and p.suffix in VALID_IMAGE_EXTS]
    labels_map = {p.stem: p for p in all_files if p.is_file() and p.suffix == ".txt" and not p.name.startswith("README")}
    
    valid_samples = []
    skipped_count = 0
    
    for img_p in image_paths:
        # Check image can be opened
        img = cv2.imread(str(img_p))
        if img is None:
            skipped_count += 1
            continue
            
        h, w = img.shape[:2]
        lbl_p = labels_map.get(img_p.stem)
        
        if not lbl_p or not lbl_p.exists():
            skipped_count += 1
            continue
            
        # Parse polygon label lines
        with open(lbl_p, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]
            
        instances = []
        is_valid = True
        
        for line in lines:
            parts = line.split()
            if len(parts) < 7:  # class_id + at least 3 (x, y) coordinate pairs
                is_valid = False
                break
                
            try:
                cid = int(float(parts[0]))
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                is_valid = False
                break
                
            if cid not in class_dict or len(coords) % 2 != 0:
                is_valid = False
                break
                
            # Clamp coordinates to [0.0, 1.0]
            clamped = [min(max(c, 0.0), 1.0) for c in coords]
            instances.append({
                "class_id": cid,
                "class_name": class_dict[cid],
                "polygon": clamped
            })
            
        if is_valid and len(instances) > 0:
            valid_samples.append({
                "image_path": img_p,
                "label_path": lbl_p,
                "stem": img_p.stem,
                "instances": instances,
                "width": w,
                "height": h
            })
        else:
            skipped_count += 1
            
    return valid_samples, skipped_count

dataset, skipped = discover_and_validate(SOURCE_DIR, CLASS_NAMES)

print("=" * 60)
print("DATASET VALIDATION SUMMARY")
print("=" * 60)
print(f"Valid Annotated Images : {len(dataset)}")
print(f"Skipped / Unlabeled    : {skipped}")
print("=" * 60)

DATASET VALIDATION SUMMARY
Valid Annotated Images : 1444
Skipped / Unlabeled    : 4


In [6]:
# ==============================================================================
# DETERMINISTIC 70/15/15 DATASET SPLITTING & LEAKAGE CHECK
# Shuffles the valid image samples deterministically using the locked seed
# and splits them into Train, Validation, and Test sets.
# ==============================================================================

# Deterministic shuffle
rng = random.Random(SEED)
shuffled_dataset = dataset.copy()
# Sort by stem first to ensure identical ordering across different OS platforms
shuffled_dataset.sort(key=lambda s: s["stem"])
rng.shuffle(shuffled_dataset)

n_total = len(shuffled_dataset)
n_train = int(round(n_total * TRAIN_RATIO))
n_val   = int(round(n_total * VAL_RATIO))
n_test  = n_total - n_train - n_val

train_samples = shuffled_dataset[:n_train]
val_samples   = shuffled_dataset[n_train:n_train + n_val]
test_samples  = shuffled_dataset[n_train + n_val:]

# Basic Data Leakage Verification
train_stems = {s["stem"] for s in train_samples}
val_stems   = {s["stem"] for s in val_samples}
test_stems  = {s["stem"] for s in test_samples}

leakage_train_val  = train_stems.intersection(val_stems)
leakage_train_test = train_stems.intersection(test_stems)
leakage_val_test   = val_stems.intersection(test_stems)

assert len(leakage_train_val) == 0, f"Leakage detected between Train and Val: {leakage_train_val}"
assert len(leakage_train_test) == 0, f"Leakage detected between Train and Test: {leakage_train_test}"
assert len(leakage_val_test) == 0, f"Leakage detected between Val and Test: {leakage_val_test}"

print("=" * 60)
print("SPLIT SUMMARY & LEAKAGE AUDIT")
print("=" * 60)
print(f"Train Split      : {len(train_samples)} images ({len(train_samples)/n_total:.1%})")
print(f"Validation Split : {len(val_samples)} images ({len(val_samples)/n_total:.1%})")
print(f"Test Split       : {len(test_samples)} images ({len(test_samples)/n_total:.1%})")
print(f"Total Images     : {n_total}")
print(f"Zero-Leakage     : PASSED (No overlapping image files between splits)")
print("=" * 60)

SPLIT SUMMARY & LEAKAGE AUDIT
Train Split      : 1011 images (70.0%)
Validation Split : 217 images (15.0%)
Test Split       : 216 images (15.0%)
Total Images     : 1444
Zero-Leakage     : PASSED (No overlapping image files between splits)


In [7]:
# ==============================================================================
# CLASS INSTANCE COUNTS & INITIAL DISTRIBUTION
# Computes the number of annotated leaf instances per class across all splits.
# ==============================================================================

def count_class_instances(sample_list):
    counts = Counter()
    for s in sample_list:
        for inst in s["instances"]:
            counts[inst["class_id"]] += 1
    return counts

initial_train_counts = count_class_instances(train_samples)
val_counts           = count_class_instances(val_samples)
test_counts          = count_class_instances(test_samples)
total_counts         = count_class_instances(dataset)

train_total_inst   = sum(initial_train_counts.values())
val_total_inst     = sum(val_counts.values())
test_total_inst    = sum(test_counts.values())
overall_total_inst = sum(total_counts.values())

# Summary table
summary_rows = []
for cid, cname in sorted(CLASS_NAMES.items()):
    tr_c = initial_train_counts[cid]
    va_c = val_counts[cid]
    te_c = test_counts[cid]
    to_c = total_counts[cid]
    
    tr_pct = (tr_c / train_total_inst * 100) if train_total_inst > 0 else 0
    va_pct = (va_c / val_total_inst * 100) if val_total_inst > 0 else 0
    te_pct = (te_c / test_total_inst * 100) if test_total_inst > 0 else 0
    to_pct = (to_c / overall_total_inst * 100) if overall_total_inst > 0 else 0
    
    summary_rows.append({
        "Class ID": cid,
        "Class Name": cname,
        "Train": f"{tr_c} ({tr_pct:.1f}%)",
        "Val": f"{va_c} ({va_pct:.1f}%)",
        "Test": f"{te_c} ({te_pct:.1f}%)",
        "Total": f"{to_c} ({to_pct:.1f}%)"
    })

summary_rows.append({
    "Class ID": "-",
    "Class Name": "TOTAL INSTANCES",
    "Train": f"{train_total_inst} (100%)",
    "Val": f"{val_total_inst} (100%)",
    "Test": f"{test_total_inst} (100%)",
    "Total": f"{overall_total_inst} (100%)"
})

df_dist = pd.DataFrame(summary_rows)
print("\n--- INITIAL CLASS INSTANCE COUNTS & DISTRIBUTION ---")
print(df_dist.to_string(index=False))


--- INITIAL CLASS INSTANCE COUNTS & DISTRIBUTION ---
Class ID      Class Name       Train         Val        Test       Total
       0         curling 377 (22.9%)  61 (20.7%)  70 (19.9%) 508 (22.2%)
       1         healthy 372 (22.6%)  62 (21.1%) 112 (31.9%) 546 (23.9%)
       2           spots 587 (35.7%) 121 (41.2%)  98 (27.9%) 806 (35.2%)
       3       yellowing 307 (18.7%)  50 (17.0%)  71 (20.2%) 428 (18.7%)
       - TOTAL INSTANCES 1643 (100%)  294 (100%)  351 (100%) 2288 (100%)


In [8]:
# ==============================================================================
# TARGETED DATA AUGMENTATION FOR MINORITY CLASSES
# If ENABLE_TARGETED_AUGMENTATION is True, this section identifies classes
# in the training split with low annotation counts and generates new augmented
# training samples (flips, rotations, brightness shifts) with adjusted polygon masks.
#
# IMPORTANT: Targeted augmentation is applied EXCLUSIVELY to the Training split.
# Validation and Test splits are NEVER augmented to ensure unbiased evaluation.
# ==============================================================================

def apply_polygon_transformation(instances, transform_type):
    """Transforms polygon coordinates according to the geometric transformation."""
    new_instances = []
    for inst in instances:
        coords = inst["polygon"].copy()
        new_coords = []
        
        if transform_type == "hflip":
            # Horizontal flip: x' = 1.0 - x, y' = y
            for j in range(0, len(coords), 2):
                new_coords.extend([round(1.0 - coords[j], 6), coords[j+1]])
        elif transform_type == "vflip":
            # Vertical flip: x' = x, y' = 1.0 - y
            for j in range(0, len(coords), 2):
                new_coords.extend([coords[j], round(1.0 - coords[j+1], 6)])
        elif transform_type == "hvflip":
            # 180 deg / both flips: x' = 1.0 - x, y' = 1.0 - y
            for j in range(0, len(coords), 2):
                new_coords.extend([round(1.0 - coords[j], 6), round(1.0 - coords[j+1], 6)])
        elif transform_type == "bright_contrast":
            # Photometric change only: coordinates unchanged
            new_coords = coords.copy()
            
        new_instances.append({
            "class_id": inst["class_id"],
            "class_name": inst["class_name"],
            "polygon": new_coords
        })
    return new_instances

if ENABLE_TARGETED_AUGMENTATION:
    aug_cache_dir = Path("labuyo_targeted_aug_cache")
    if aug_cache_dir.exists():
        shutil.rmtree(aug_cache_dir)
    aug_cache_dir.mkdir(parents=True, exist_ok=True)
    
    max_train_count = max(initial_train_counts.values())
    target_count = int(max_train_count * TARGET_BALANCE_RATIO)
    
    augmented_samples = []
    aug_counter = 0
    transform_types = ["hflip", "vflip", "hvflip", "bright_contrast"]
    
    print(f"[TARGETED AUGMENTATION] Balancing minority classes up to target: ~{target_count} instances per class...")
    
    for cid, cname in CLASS_NAMES.items():
        curr_count = initial_train_counts[cid]
        deficit = target_count - curr_count
        
        if deficit <= 0:
            print(f" - Class '{cname}' (ID {cid}): {curr_count} instances (Sufficient, no extra augmentation needed)")
            continue
            
        # Collect candidate training images containing this minority class
        candidates = [s for s in train_samples if any(inst["class_id"] == cid for inst in s["instances"])]
        if not candidates:
            continue
            
        print(f" - Class '{cname}' (ID {cid}): {curr_count} instances (Deficit: {deficit}). Generating targeted augmentations...")
        
        generated_for_class = 0
        cand_idx = 0
        
        while generated_for_class < deficit:
            src_sample = candidates[cand_idx % len(candidates)]
            ttype = transform_types[(generated_for_class) % len(transform_types)]
            
            orig_img = cv2.imread(str(src_sample["image_path"]))
            if orig_img is None:
                cand_idx += 1
                continue
                
            # Apply image transformation
            if ttype == "hflip":
                aug_img = cv2.flip(orig_img, 1)
            elif ttype == "vflip":
                aug_img = cv2.flip(orig_img, 0)
            elif ttype == "hvflip":
                aug_img = cv2.flip(orig_img, -1)
            elif ttype == "bright_contrast":
                alpha = random.uniform(0.85, 1.15)  # Contrast
                beta = random.randint(-15, 15)      # Brightness
                aug_img = cv2.convertScaleAbs(orig_img, alpha=alpha, beta=beta)
                
            aug_instances = apply_polygon_transformation(src_sample["instances"], ttype)
            
            aug_stem = f"{src_sample['stem']}_aug_{ttype}_{aug_counter}"
            aug_img_path = aug_cache_dir / f"{aug_stem}.jpg"
            aug_lbl_path = aug_cache_dir / f"{aug_stem}.txt"
            
            cv2.imwrite(str(aug_img_path), aug_img)
            
            # Write new YOLO polygon label file
            with open(aug_lbl_path, "w", encoding="utf-8") as f:
                for inst in aug_instances:
                    coords_str = " ".join(f"{c:.6f}" for c in inst["polygon"])
                    f.write(f"{inst['class_id']} {coords_str}\n")
                    
            augmented_samples.append({
                "image_path": aug_img_path,
                "label_path": aug_lbl_path,
                "stem": aug_stem,
                "instances": aug_instances,
                "width": src_sample["width"],
                "height": src_sample["height"]
            })
            
            # Count minority instances added
            minority_added = sum(1 for inst in aug_instances if inst["class_id"] == cid)
            generated_for_class += minority_added
            aug_counter += 1
            cand_idx += 1
            
    # Combine original training samples with newly generated targeted augmentations
    final_train_samples = train_samples + augmented_samples
    print(f"\n[TARGETED AUGMENTATION COMPLETE] Generated {len(augmented_samples)} new training images.")
    print(f"Final Training Set Size: {len(final_train_samples)} images (Original: {len(train_samples)})")
else:
    final_train_samples = train_samples.copy()
    print("[TARGETED AUGMENTATION] Bypassed (ENABLE_TARGETED_AUGMENTATION = False). Using original train split.")

# Compare Before vs After Balance Table
post_train_counts = count_class_instances(final_train_samples)
balance_rows = []
for cid, cname in sorted(CLASS_NAMES.items()):
    b_c = initial_train_counts[cid]
    a_c = post_train_counts[cid]
    balance_rows.append({
        "Class ID": cid,
        "Class Name": cname,
        "Before Augmentation": b_c,
        "After Targeted Aug": a_c,
        "Net Gain": f"+{a_c - b_c}"
    })
print("\n--- TRAINING CLASS BALANCE COMPARISON ---")
print(pd.DataFrame(balance_rows).to_string(index=False))

[TARGETED AUGMENTATION] Bypassed (ENABLE_TARGETED_AUGMENTATION = False). Using original train split.

--- TRAINING CLASS BALANCE COMPARISON ---
 Class ID Class Name  Before Augmentation  After Targeted Aug Net Gain
        0    curling                  377                 377       +0
        1    healthy                  372                 372       +0
        2      spots                  587                 587       +0
        3  yellowing                  307                 307       +0


In [9]:
# ==============================================================================
# YOLO DATASET GENERATION & data.yaml CREATION
# Copies the split images and label files into a clean YOLO folder structure:
#   labuyo_yolo_dataset/
#   ├── images/ (train, val, test)
#   ├── labels/ (train, val, test)
#   └── data.yaml
# ==============================================================================

def create_yolo_dataset(splits_dict: dict, out_dir: Path, class_dict: dict):
    if out_dir.exists():
        shutil.rmtree(out_dir)
        
    for split_name in ["train", "val", "test"]:
        (out_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)
        
    for split_name, samples in splits_dict.items():
        for s in samples:
            # Copy image
            shutil.copy2(s["image_path"], out_dir / "images" / split_name / s["image_path"].name)
            # Copy label
            shutil.copy2(s["label_path"], out_dir / "labels" / split_name / s["label_path"].name)
            
    # Create data.yaml
    yaml_data = {
        "path": str(out_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(class_dict),
        "names": {int(k): str(v) for k, v in class_dict.items()}
    }
    
    yaml_path = out_dir / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(yaml_data, f, sort_keys=False)
        
    print(f"[DATASET] Generated YOLO dataset at: {out_dir.resolve()}")
    print(f"[DATASET] Generated data.yaml at: {yaml_path.resolve()}")
    return yaml_path

splits = {
    "train": final_train_samples,
    "val": val_samples,
    "test": test_samples
}

DATA_YAML = create_yolo_dataset(splits, OUTPUT_DIR, CLASS_NAMES)

print("\n--- GENERATED data.yaml ---")
with open(DATA_YAML, "r") as f:
    print(f.read())

[DATASET] Generated YOLO dataset at: C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5
[DATASET] Generated data.yaml at: C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5\data.yaml

--- GENERATED data.yaml ---
path: C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5
train: images/train
val: images/val
test: images/test
nc: 4
names:
  0: curling
  1: healthy
  2: spots
  3: yellowing



In [10]:
# ==============================================================================
# MODEL INITIALIZATION & TRAINING CONFIGURATION
# Loads the YOLO26n instance-segmentation model architecture and configures training arguments.
# ==============================================================================

# 1. Initialize YOLO26n Segmentation Model
print(f"[MODEL] Initializing YOLO26n instance segmentation model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)

# 2. Training Hyperparameters
# Based on the current Ultralytics YOLO training defaults.
# Explicit values are documented here for reproducibility.

TRAIN_CONFIG = {

    # --------------------------------------------------------
    # DATA & TRAINING CONTROL
    # --------------------------------------------------------
    "data": str(DATA_YAML.resolve()),

    "epochs": 100,
    "time": None,

    # 25 is reasonable for this dataset size
    "patience": 25,

    # RTX 2050 4 GB
    "batch": 8,
    "imgsz": 640,

    "device": 0 if torch.cuda.is_available() else "cpu",

    "workers": 4,

    # Use all images
    "fraction": 1.0,

    # --------------------------------------------------------
    # OUTPUT / EXPERIMENT MANAGEMENT
    # --------------------------------------------------------
    "project": str(RUNS_DIR),
    "name": EXPERIMENT_NAME,

    "exist_ok": False,
    "save_dir": None,

    "save": True,
    "save_period": -1,

    "plots": True,
    "verbose": True,

    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------
    "seed": SEED,
    "deterministic": True,

    # --------------------------------------------------------
    # MODEL / OPTIMIZER
    # --------------------------------------------------------
    "optimizer": "auto",

    # IMPORTANT:
    # When optimizer="auto", Ultralytics determines
    # the optimizer/LR automatically.
    "lr0": 0.01,
    "lrf": 0.01,

    "momentum": 0.937,
    "weight_decay": 0.0005,

    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,

    "cos_lr": False,

    # --------------------------------------------------------
    # TRAINING PRECISION / PERFORMANCE
    # --------------------------------------------------------
    "amp": True,

    # Keep False initially.
    # Caching may consume substantial system RAM.
    "cache": False,

    "rect": False,
    "multi_scale": 0.0,

    "profile": False,

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------
    "val": True,

    # Validate on validation split during training
    "split": "val",

    # --------------------------------------------------------
    # INSTANCE SEGMENTATION
    # --------------------------------------------------------
    "overlap_mask": True,
    "mask_ratio": 4,

    # --------------------------------------------------------
    # LOSS WEIGHTS
    # --------------------------------------------------------
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,

    # Nominal batch size used for loss normalization
    "nbs": 64,

    # --------------------------------------------------------
    # CLASS IMBALANCE
    # --------------------------------------------------------
    "cls_pw": 0.0,

    # --------------------------------------------------------
    # REGULARIZATION
    # --------------------------------------------------------
    "dropout": 0.0,

    # --------------------------------------------------------
    # COLOR AUGMENTATION
    # --------------------------------------------------------
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,

    # --------------------------------------------------------
    # GEOMETRIC AUGMENTATION
    # --------------------------------------------------------
    "degrees": 10.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,

    # Leaves can naturally appear in many orientations,
    # but vertical flipping may be less biologically realistic.
    "flipud": 0.0,
    "fliplr": 0.5,

    # Preserve natural RGB appearance
    "bgr": 0.0,

    # --------------------------------------------------------
    # COMPOSITION AUGMENTATION
    # --------------------------------------------------------
    "mosaic": 1.0,

    # Keep these conservative for your dataset.
    "mixup": 0.0,
    "cutmix": 0.0,

    # Segmentation-specific augmentation
    "copy_paste": 0.0,
    "copy_paste_mode": "flip",

    # --------------------------------------------------------
    # MOSAIC STABILIZATION
    # --------------------------------------------------------
    "close_mosaic": 10,

    # --------------------------------------------------------
    # OTHER
    # --------------------------------------------------------
    "resume": False,
    "freeze": None,
    "single_cls": False,
    "classes": None,
}

print("\n--- TRAINING CONFIGURATION (YOLO26n-seg) ---")
for k, v in TRAIN_CONFIG.items():
    print(f" {k:<12}: {v}")

[MODEL] Initializing YOLO26n instance segmentation model: yolo26n-seg.pt

--- TRAINING CONFIGURATION (YOLO26n-seg) ---
 data        : C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5\data.yaml
 epochs      : 100
 time        : None
 patience    : 25
 batch       : 8
 imgsz       : 640
 device      : 0
 workers     : 4
 fraction    : 1.0
 project     : labuyo_segmentation_v6
 name        : yolo26n_labuyo_leaf_seg_v6
 exist_ok    : False
 save_dir    : None
 save        : True
 save_period : -1
 plots       : True
 verbose     : True
 seed        : 42
 deterministic: True
 optimizer   : auto
 lr0         : 0.01
 lrf         : 0.01
 momentum    : 0.937
 weight_decay: 0.0005
 warmup_epochs: 3.0
 warmup_momentum: 0.8
 warmup_bias_lr: 0.1
 cos_lr      : False
 amp         : True
 cache       : False
 rect        : False
 multi_scale : 0.0
 profile     : False
 val         : True
 split       : val
 overlap_mask: True
 mask_ratio  : 4
 box         : 

In [11]:
# ==============================================================================
# RUN MODEL TRAINING
# Executes the training process. Best weights will be saved to:
#   runs/labuyo_segmentation/yolo26n_labuyo_leaf_seg/weights/best.pt
# ==============================================================================

results = model.train(**TRAIN_CONFIG)
print("[INFO] Ready to train YOLO26n! Run: results = model.train(**TRAIN_CONFIG)")

New https://pypi.org/project/ultralytics/8.4.138 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.131  Python-3.10.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\CELL27\Pictures\siliguide_segmenter\siling_labuyo_cnn_model\labuyo_yolo_dataset_v5\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, l

In [12]:
# ==============================================================================
# MODEL EVALUATION (ACCURACY, PRECISION, RECALL, F1, mAP & CONFUSION MATRIX)
# Evaluates the best trained model on the held-out Test split and computes:
# - Precision (P), Recall (R), F1-Score, and mAP (Box & Mask)
# - Per-class performance breakdown table
# - Overall accuracy from the confusion matrix
# - Confusion matrix plot (normalized & raw)
# ==============================================================================

RUN_DIR = Path("runs/segment/runs/labuyo_segmentation_v2")

best_weights = RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"

if best_weights.exists():
    print(f"[EVAL] Loading trained model from: {best_weights}")
    trained_model = YOLO(str(best_weights))
    
    # Run evaluation on held-out Test set
    eval_yaml = str(DATA_YAML.resolve()) if 'DATA_YAML' in globals() and DATA_YAML.exists() else "leaf_segmenter_v1/data.yaml"
    test_metrics = trained_model.val(data=eval_yaml, split="test")
    
    # 1. Overall Mask Metrics
    mask_p = float(test_metrics.seg.mp)
    mask_r = float(test_metrics.seg.mr)
    mask_f1 = (2 * mask_p * mask_r / (mask_p + mask_r + 1e-9)) if (mask_p + mask_r) > 0 else 0.0
    mask_map50 = float(test_metrics.seg.map50)
    mask_map = float(test_metrics.seg.map)
    
    # 2. Overall Box Metrics
    box_p = float(test_metrics.box.mp)
    box_r = float(test_metrics.box.mr)
    box_f1 = (2 * box_p * box_r / (box_p + box_r + 1e-9)) if (box_p + box_r) > 0 else 0.0
    box_map50 = float(test_metrics.box.map50)
    box_map = float(test_metrics.box.map)
    
    # 3. Overall Accuracy from Confusion Matrix
    cm_matrix = test_metrics.confusion_matrix.matrix
    total_preds = np.sum(cm_matrix)
    correct_preds = np.trace(cm_matrix)
    accuracy = (correct_preds / total_preds) if total_preds > 0 else 0.0
    
    print("\n" + "=" * 65)
    print(f"Overall Accuracy (from CM) : {accuracy:.4f} ({accuracy:.2%})")
    print("-" * 65)
    print(f"Mask Precision (P)         : {mask_p:.4f}")
    print(f"Mask Recall (R)            : {mask_r:.4f}")
    print(f"Mask F1-Score              : {mask_f1:.4f}")
    print(f"Mask mAP @ 0.50            : {mask_map50:.4f}")
    print(f"Mask mAP @ 0.50:0.95       : {mask_map:.4f}")
    print("-" * 65)
    print(f"Box Precision (P)          : {box_p:.4f}")
    print(f"Box Recall (R)             : {box_r:.4f}")
    print(f"Box F1-Score               : {box_f1:.4f}")
    print(f"Box mAP @ 0.50             : {box_map50:.4f}")
    print(f"Box mAP @ 0.50:0.95        : {box_map:.4f}")
    print("=" * 65)
    
    # 4. Per-Class Breakdown
    per_class_data = []
    for cid, cname in CLASS_NAMES.items():
        if cid < len(test_metrics.seg.maps):
            p_c = float(test_metrics.seg.p[cid])
            r_c = float(test_metrics.seg.r[cid])
            f1_c = (2 * p_c * r_c / (p_c + r_c + 1e-9)) if (p_c + r_c) > 0 else 0.0
            map_c = float(test_metrics.seg.maps[cid])
            per_class_data.append({
                "Class ID": cid,
                "Class Name": cname,
                "Precision": f"{p_c:.4f}",
                "Recall": f"{r_c:.4f}",
                "F1-Score": f"{f1_c:.4f}",
                "Mask mAP50-95": f"{map_c:.4f}"
            })
            
    if per_class_data:
        print("\n--- PER-CLASS PERFORMANCE BREAKDOWN ---")
        print(pd.DataFrame(per_class_data).to_string(index=False))
        
    # 5. Display Confusion Matrix
    save_dir = Path(test_metrics.save_dir)
    cm_paths = [save_dir / "confusion_matrix_normalized.png", save_dir / "confusion_matrix.png"]
    for cm_img_p in cm_paths:
        if cm_img_p.exists():
            cm_img = cv2.imread(str(cm_img_p))
            cm_img = cv2.cvtColor(cm_img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(8, 6), dpi=100)
            plt.imshow(cm_img)
            plt.title(f"Confusion Matrix: {cm_img_p.name}", fontsize=12, fontweight="bold")
            plt.axis("off")
            plt.show()
            break
else:
    print(f"[INFO] '{best_weights}' not found yet. Train the model first to run evaluation.")

[INFO] 'labuyo_segmentation_v6\yolo26n_labuyo_leaf_seg_v6\weights\best.pt' not found yet. Train the model first to run evaluation.


In [13]:
# ==============================================================================
# PREDICTION VISUALIZATION
# Runs inference on sample test images and displays predicted leaf masks.
# ==============================================================================

def preview_test_predictions(model_obj, samples, conf=0.4):
    num_previews = min(3, len(samples))
    fig, axes = plt.subplots(num_previews, 2, figsize=(14, 5 * num_previews))
    if num_previews == 1:
        axes = np.array([axes])
        
    for i in range(num_previews):
        s = samples[i]
        img_path = str(s["image_path"])
        
        # Ground Truth
        gt_img = cv2.imread(img_path)
        gt_img = cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB)
        h, w = gt_img.shape[:2]
        overlay = gt_img.copy()
        for inst in s["instances"]:
            coords = inst["polygon"]
            pts = np.array([[int(coords[j] * w), int(coords[j+1] * h)] for j in range(0, len(coords), 2)], np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(overlay, [pts], (46, 204, 113))
            cv2.polylines(gt_img, [pts], True, (46, 204, 113), 2)
        gt_blended = cv2.addWeighted(overlay, 0.4, gt_img, 0.6, 0)
        
        axes[i, 0].imshow(gt_blended)
        axes[i, 0].set_title(f"Ground Truth: {s['image_path'].name}", fontsize=11, fontweight="bold")
        axes[i, 0].axis("off")
        
        # Prediction
        res = model_obj.predict(source=img_path, conf=conf, verbose=False)[0]
        pred_rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        
        n_dets = len(res.boxes) if res.boxes is not None else 0
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title(f"YOLO26n Prediction ({n_dets} leaves detected, conf>={conf})", fontsize=11, fontweight="bold")
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()

if best_weights.exists() and len(test_samples) > 0:
    print("[VISUALIZATION] Rendering predictions on test images...")
    preview_test_predictions(trained_model, test_samples)
else:
    print("[INFO] Prediction visualizer will activate after model training completes.")

[INFO] Prediction visualizer will activate after model training completes.
